In [1]:
import os
import threading
import tkinter as tk
from tkinter import messagebox

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    MBartForConditionalGeneration,
    MBart50Tokenizer,
)

# ------------------------------------------------------------
# CONFIG: setting paths
# ------------------------------------------------------------
BASE_DIR = r"C:\Users\HP\Unified Bangla"  
SAVED_MODELS = os.path.join(BASE_DIR, "saved_models1")

DIALECT_CLS_DIR = os.path.join(SAVED_MODELS, "banglabert_dialect")
MBART_DIR       = os.path.join(SAVED_MODELS, "mbart50_best")

SUPPORTED_DIALECTS = ["bangla", "barishal", "chittagong", "mymensingh", "noakhali", "sylhet"]
MBART_LANG = "bn_IN"

# UI labels (Bangla + English)
DIALECT_UI = {
    "bangla":      ("শুদ্ধ বাংলা", "Standard Bangla"),
    "barishal":    ("বরিশাল", "Barishal"),
    "chittagong":  ("চট্টগ্রাম", "Chittagong"),
    "mymensingh":  ("ময়মনসিংহ", "Mymensingh"),
    "noakhali":    ("নোয়াখালী", "Noakhali"),
    "sylhet":      ("সিলেট", "Sylhet"),
}

# ------------------------------------------------------------
# LOADING MODELS 
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

cls_tokenizer = AutoTokenizer.from_pretrained(DIALECT_CLS_DIR, use_fast=True)
cls_model = AutoModelForSequenceClassification.from_pretrained(DIALECT_CLS_DIR).to(device)
cls_model.eval()
id2label = cls_model.config.id2label

mbart_tokenizer = MBart50Tokenizer.from_pretrained(MBART_DIR)
mbart_model = MBartForConditionalGeneration.from_pretrained(MBART_DIR).to(device)
mbart_model.eval()

mbart_tokenizer.src_lang = MBART_LANG
mbart_tokenizer.tgt_lang = MBART_LANG
if hasattr(mbart_tokenizer, "lang_code_to_id"):
    mbart_model.config.forced_bos_token_id = mbart_tokenizer.lang_code_to_id[MBART_LANG]


@torch.no_grad()
def predict_dialect(sentence: str) -> str:
    sentence = sentence.strip()
    if not sentence:
        return ""
    inputs = cls_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)
    logits = cls_model(**inputs).logits
    pred_id = int(torch.argmax(logits, dim=-1).item())
    return id2label.get(pred_id, str(pred_id)).strip().lower()


@torch.no_grad()
def translate_to_target(sentence: str, src: str, tgt: str) -> str:
    sentence = sentence.strip()
    src = src.strip().lower()
    tgt = tgt.strip().lower()

    if src not in SUPPORTED_DIALECTS:
        raise ValueError(
            f"Classifier predicted '{src}', but SUPPORTED_DIALECTS={SUPPORTED_DIALECTS}.\n"
            f"Fix id2label mapping or SUPPORTED_DIALECTS."
        )
    if tgt not in SUPPORTED_DIALECTS:
        raise ValueError(f"Target '{tgt}' not in SUPPORTED_DIALECTS={SUPPORTED_DIALECTS}")

    inp = f"<{src}> {sentence} <{tgt}>"
    enc = mbart_tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    ).to(device)

    out_ids = mbart_model.generate(
        **enc,
        max_length=96,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3
    )
    return mbart_tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


# ------------------------------------------------------------
# GUI
# ------------------------------------------------------------
class UnifiedBanglaApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Unified Bangla")
        self.geometry("720x560")   # compact
        self.configure(bg="white")

        # compact fonts (your “perfect size” direction)
        self.font_header_bn = ("Noto Sans Bengali", 16, "bold")
        self.font_header_en = ("Segoe UI", 9)

        self.font_title_bn = ("Noto Sans Bengali", 13)
        self.font_title_en = ("Segoe UI", 8)

        self.font_box = ("Noto Sans Bengali", 9)
        self.font_btn = ("Noto Sans Bengali", 9)

        self.detected_src = None
        self.selected_tgt = None
        self.tgt_buttons = {}  # key -> button

        self._build_ui()

    def _build_ui(self):
        # Header bar 
        header = tk.Frame(self, bg="#DDF5EA", height=70, bd=1, relief="solid")
        header.pack(fill="x")
        header.pack_propagate(False)

        # ---- Bangla title with two colors ----
        title_frame = tk.Frame(header, bg="#DDF5EA")
        title_frame.pack(pady=(10, 0))

        tk.Label(
            title_frame,
            text="ইউনিফাইড ",
            bg="#DDF5EA",
            fg="#0A7C4C",   # green
            font=self.font_header_bn
        ).pack(side="left")

        tk.Label(
            title_frame,
            text="বাংলা",
            bg="#DDF5EA",
            fg="#EC2238",      # red
            font=self.font_header_bn
        ).pack(side="left")

        # English subtitle 
        tk.Label(
            header,
            text="(Unified Bangla)",
            bg="#DDF5EA",
            fg="black",
            font=self.font_header_en
        ).pack()


        body = tk.Frame(self, bg="white")
        body.pack(fill="both", expand=True, padx=14, pady=10)

        # Input section
        self._section(body, "নির্ধারিত বাক্যটি লিখুন", "Enter an input sentence")
        self.input_entry = tk.Entry(body, font=self.font_box, bd=1, relief="solid")
        self.input_entry.pack(fill="x", pady=(4, 6), ipady=0)  # small height

        self.detect_btn = tk.Button(
            body, text="এখানে চাপুন (Click here)", font=self.font_btn,
            command=self.on_detect_clicked
        )
        self.detect_btn.pack(anchor="w", pady=(0, 8))

        # Status (small)
        self.status_var = tk.StringVar(value="")
        tk.Label(body, textvariable=self.status_var, bg="white", fg="#555",
                 font=("Segoe UI", 8)).pack(anchor="w", pady=(0, 8))

        # Detected dialect section
        self._section(body, "শনাক্তকারী আঞ্চলিক ভাষা", "Detected dialect")
        self.detected_var = tk.StringVar(value="")
        self.detected_entry = tk.Entry(
            body, font=self.font_box, bd=1, relief="solid",
            textvariable=self.detected_var, state="readonly"
        )
        self.detected_entry.pack(fill="x", pady=(4, 10), ipady=0)

        # Target + Output (hidden until detection)
        self.target_block = tk.Frame(body, bg="white")
        self.target_block.pack_forget()

        self._section(self.target_block, "কাঙ্ক্ষিত আঞ্চলিক ভাষা নির্বাচন করুন", "Select target dialect")

        # “Radio-like” buttons grid
        self.btn_grid = tk.Frame(self.target_block, bg="white")
        self.btn_grid.pack(anchor="w", pady=(6, 8))

        # 2x3 layout 
        grid = [
            ["bangla", "barishal", "chittagong"],
            ["mymensingh", "noakhali", "sylhet"],
        ]

        for r, row in enumerate(grid):
            for c, key in enumerate(row):
                bn, en = DIALECT_UI[key]
                text = f"{bn}\n{en}"
                b = tk.Button(
                    self.btn_grid,
                    text=text,
                    font=("Noto Sans Bengali", 9),
                    width=14,
                    height=2,
                    bd=1,
                    relief="solid",
                    bg="white",
                    command=lambda k=key: self.on_target_clicked(k)
                )
                b.grid(row=r, column=c, padx=6, pady=6, sticky="w")
                self.tgt_buttons[key] = b

        # Output section (after target buttons)
        self._section(self.target_block, "অনুবাদিত বাক্য", "Translated Sentence")
        self.output_text = tk.Text(self.target_block, height=2, font=self.font_box, bd=1, relief="solid", wrap="word")
        self.output_text.pack(fill="x", pady=(4, 0))

    def _section(self, parent, bn, en):
        tk.Label(parent, text=bn, bg="white", fg="black", font=self.font_title_bn).pack(anchor="w")
        tk.Label(parent, text=en, bg="white", fg="black", font=self.font_title_en).pack(anchor="w")

    # ---------------------------
    # Actions
    # ---------------------------
    def on_detect_clicked(self):
        sentence = self.input_entry.get().strip()
        if not sentence:
            messagebox.showinfo("Info", "বাক্য লিখুন\nPlease enter a sentence")
            return

        self.detect_btn.config(state="disabled")
        self.status_var.set("Detecting...")

        threading.Thread(target=self._detect_worker, args=(sentence,), daemon=True).start()

    def _detect_worker(self, sentence: str):
        try:
            src = predict_dialect(sentence)
            if not src:
                raise ValueError("No dialect predicted.")
        except Exception as e:
            err = str(e)  # ✅ capture text to avoid NameError
            self.after(0, lambda err=err: (
                self.status_var.set(""),
                self.detect_btn.config(state="normal"),
                messagebox.showerror("Detection error", err)
            ))
            return

        def update_ui():
            self.detected_src = src
            bn, en = DIALECT_UI.get(src, (src, "Unknown label"))
            self.detected_var.set(f"{bn} ({en})")

            # Show target + output block now
            self.target_block.pack(fill="x", pady=(4, 0))

            # default select first target
            self.on_target_clicked(self.selected_tgt or "bangla")

            self.status_var.set("")
            self.detect_btn.config(state="normal")

        self.after(0, update_ui)

    def on_target_clicked(self, tgt_key: str):
        # behaves like radio: one selected at a time
        self.selected_tgt = tgt_key
        for k, btn in self.tgt_buttons.items():
            btn.config(bg="#DDF5EA" if k == tgt_key else "white")

        # auto-translate
        if not self.detected_src:
            return
        sentence = self.input_entry.get().strip()
        if not sentence:
            return

        self.status_var.set("Translating...")
        threading.Thread(
            target=self._translate_worker,
            args=(sentence, self.detected_src, tgt_key),
            daemon=True
        ).start()

    def _translate_worker(self, sentence: str, src: str, tgt: str):
        try:
            out = translate_to_target(sentence, src, tgt)
        except Exception as e:
            err = str(e)  # ✅ capture text to avoid NameError
            self.after(0, lambda err=err: (
                self.status_var.set(""),
                messagebox.showerror("Translation error", err)
            ))
            return

        def update():
            self.output_text.delete("1.0", "end")
            self.output_text.insert("1.0", out)
            self.status_var.set("")

        self.after(0, update)


if __name__ == "__main__":
    app = UnifiedBanglaApp()
    app.mainloop()


C:\Users\HP\anaconda3\envs\banglabert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
